# Module 2: RAG Systems (Retrieval-Augmented Generation)

**Goal:** Build a working RAG pipeline from scratch — from documents to answerable questions.

**What you'll learn:**
1. Why RAG exists and when to use it
2. Document chunking strategies (fixed-size vs. semantic)
3. Creating and querying a vector store (ChromaDB)
4. Building a complete Q&A pipeline with source citations — in plain Python, no framework
5. Evaluating RAG quality

**Prerequisites:** Module 01 completed. `.env` file with `OPENAI_API_KEY` (for embeddings) and the key for whichever chat model you use.

---

## Why RAG?

LLMs have a **knowledge cutoff** and no access to your private data. RAG solves this by:

```
User question
     │
     ▼
[Retriever] ──searches──▶ Vector DB (your documents)
     │                         │
     │◀──── top-k chunks ──────┘
     │
     ▼
[Generator] ──reads──▶ LLM (context + question → answer)
     │
     ▼
Answer + sources
```

**Use RAG when:** domain-specific knowledge, current events, private documents, reducing hallucinations.

## 0. Setup

In [ ]:
%pip install -q openai chromadb tiktoken python-dotenv

In [ ]:
import os
import sys
from dotenv import load_dotenv

# The provider layer (llm/) and .env live at the repo root, two folders up.
sys.path.insert(0, "../..")
load_dotenv(dotenv_path="../../.env")

from llm import ask, default_model

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in ../../.env (used for embeddings)"
print(f"✅ Environment ready — answers from {default_model()}")

---
## 1. Documents: Our Knowledge Base

In production these come from PDFs, databases, web pages, etc. Here we use in-memory documents.

In [ ]:
documents = [
    {
        "id": "rag_intro",
        "content": """Retrieval-Augmented Generation (RAG) combines information retrieval with 
        large language models. It retrieves relevant documents from a knowledge base and 
        provides them as context to the LLM, enabling more accurate and grounded responses. 
        RAG was introduced by Lewis et al. in 2020 and has become the dominant architecture 
        for knowledge-intensive NLP tasks.""",
        "metadata": {"source": "rag_overview", "topic": "fundamentals"}
    },
    {
        "id": "vector_db",
        "content": """Vector databases store embeddings — dense numerical representations of text. 
        When you search, your query is also embedded and the database returns the most 
        semantically similar stored vectors using approximate nearest-neighbor (ANN) search. 
        Popular options: ChromaDB (local), Pinecone (managed), Weaviate (open-source), 
        pgvector (PostgreSQL extension).""",
        "metadata": {"source": "vector_db_guide", "topic": "infrastructure"}
    },
    {
        "id": "chunking",
        "content": """Chunking splits documents into smaller pieces before embedding. 
        Fixed-size chunking splits every N tokens regardless of content boundaries. 
        Semantic chunking splits at natural boundaries (sentences, paragraphs, sections). 
        Chunk size matters: too small loses context, too large dilutes relevance. 
        A common starting point is 512 tokens with 10% overlap between chunks.""",
        "metadata": {"source": "chunking_guide", "topic": "preprocessing"}
    },
    {
        "id": "embeddings",
        "content": """Embeddings are vectors that capture semantic meaning. Two sentences about 
        the same topic have similar (high cosine similarity) embeddings even if they use 
        different words. Popular embedding models: OpenAI text-embedding-3-small (fast, cheap), 
        text-embedding-3-large (best quality), sentence-transformers/all-MiniLM-L6-v2 (free, local).""",
        "metadata": {"source": "embeddings_guide", "topic": "fundamentals"}
    },
    {
        "id": "advanced_rag",
        "content": """Advanced RAG techniques include: HyDE (Hypothetical Document Embeddings) 
        which generates a hypothetical answer and uses that for retrieval; Re-ranking which 
        retrieves more candidates then reranks with a cross-encoder; Hybrid search which 
        combines BM25 keyword search with vector similarity; Query expansion which rewrites 
        the question multiple ways to improve recall.""",
        "metadata": {"source": "advanced_rag", "topic": "advanced"}
    },
]

print(f"Loaded {len(documents)} documents")
print(f"Example document: '{documents[0]['content'][:100]}...'")

---
## 2. Chunking

Compare fixed-size vs. recursive character splitting. **Overlap** ensures context isn't lost at chunk boundaries.

In [ ]:
def fixed_size_chunks(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Strategy 1: every chunk_size characters, ignoring sentence boundaries."""
    step = chunk_size - overlap
    return [text[i:i + chunk_size] for i in range(0, len(text), step)]


def recursive_chunks(text: str, chunk_size: int, overlap: int, separators=(". ", " ")) -> list[str]:
    """Strategy 2: pack whole sentences (then words, if a sentence is too long) up to chunk_size."""
    for sep in separators:
        units = text.split(sep)
        if all(len(u) <= chunk_size for u in units):
            break
    chunks, current = [], ""
    for unit in units:
        candidate = f"{current}{sep}{unit}" if current else unit
        if len(candidate) <= chunk_size or not current:
            current = candidate
            continue
        chunks.append(current)
        carried = f"{current[-overlap:]}{sep}{unit}" if overlap else unit
        current = carried if len(carried) <= chunk_size else unit
    if current:
        chunks.append(current)
    return chunks


def chunk_all(docs, splitter, **kwargs):
    return [
        {"id": f"{doc['id']}-{n}", "text": piece, "metadata": doc["metadata"]}
        for doc in docs
        for n, piece in enumerate(splitter(" ".join(doc["content"].split()), **kwargs))
    ]


fixed = chunk_all(documents, fixed_size_chunks, chunk_size=200, overlap=20)
recursive = chunk_all(documents, recursive_chunks, chunk_size=300, overlap=30)

print(f"Original docs: {len(documents)}")
print(f"Fixed-size chunks: {len(fixed)}")
print(f"Recursive chunks: {len(recursive)}")
print(f"\nExample chunk (fixed-size — note the cut mid-word):\n{fixed[1]['text']}")
print(f"\nExample chunk (recursive):\n{recursive[0]['text']}")

In [ ]:
# 🧪 EXERCISE: Experiment with chunk_size values: 100, 500, 1000
# Print chunk counts and check if sentences are split mid-way.
# Which strategy preserves meaning better?

---
## 3. Vector Store (ChromaDB)

Embed all chunks and store them. ChromaDB stores to disk by default — no server needed for development.

In [ ]:
import shutil
import chromadb
from openai import OpenAI

# Embeddings are not part of the provider layer; this page calls OpenAI's directly.
embedding_client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"

def embed(texts: list[str]) -> list[list[float]]:
    response = embedding_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

# Clean up any previous run
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.create_collection("rag_tutorial", metadata={"hnsw:space": "cosine"})

# Build the vector store from the recursive chunks
collection.add(
    ids=[c["id"] for c in recursive],
    documents=[c["text"] for c in recursive],
    metadatas=[c["metadata"] for c in recursive],
    embeddings=embed([c["text"] for c in recursive]),
)

print(f"✅ Vector store created with {collection.count()} vectors")

---
## 4. Retrieval: Finding Relevant Chunks

In [ ]:
def retrieve(query: str, k: int = 3) -> list[dict]:
    """Top-k chunks for the query, with their metadata and cosine distance."""
    result = collection.query(query_embeddings=embed([query]), n_results=k)
    return [
        {"text": text, "metadata": meta, "distance": distance}
        for text, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0])
    ]

query = "What are vector databases and which ones are popular?"
results = retrieve(query)

print(f"Query: {query}")
print(f"\nTop {len(results)} retrieved chunks:")
for i, chunk in enumerate(results, 1):
    print(f"\n[{i}] Source: {chunk['metadata'].get('source', 'unknown')}")
    print(f"    Content: {chunk['text'][:200]}...")

In [ ]:
# Similarity search with scores (lower distance = more similar)
print("Results with similarity scores:")
for chunk in retrieve(query, k=3):
    print(f"  Distance: {chunk['distance']:.4f} | {chunk['text'][:80]}...")

# 🔑 Key insight: cosine distances below ~0.3 are typically good matches

---
## 5. Generation: Full RAG Chain

Combine retrieval + LLM into an end-to-end pipeline: retrieve, format the context, ask. The prompt explicitly tells the model to use only the provided context.

In [ ]:
RAG_SYSTEM = """You are a helpful assistant. Answer the question using ONLY the provided context.
If the context doesn't contain the answer, say "I don't have information about that in my knowledge base."
Always cite which source(s) you used at the end."""

RAG_PROMPT = """Context:
{context}

Question: {question}

Answer:"""

def format_chunks(chunks):
    return "\n\n---\n\n".join(
        f"[Source: {c['metadata'].get('source', 'unknown')}]\n{c['text']}" for c in chunks
    )

def rag_answer(question: str, k: int = 3) -> str:
    context = format_chunks(retrieve(question, k=k))
    return ask(RAG_PROMPT.format(context=context, question=question), system=RAG_SYSTEM)

print("✅ RAG pipeline built")

In [ ]:
# Test the full chain
test_questions = [
    "What is RAG and who invented it?",
    "What chunk size should I use?",
    "What is HyDE?",
    "What is the capital of France?",  # Out-of-scope — observe the fallback
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"A: {rag_answer(question)}")

---
## 6. Evaluating RAG Quality

Three key metrics for RAG evaluation:
- **Context relevance**: Did we retrieve the right chunks?
- **Faithfulness**: Is the answer grounded in the retrieved context (no hallucination)?
- **Answer relevance**: Does the answer actually address the question?

In [ ]:
import json

JUDGE_SYSTEM = "You are a strict RAG evaluation judge. Respond with one JSON object and nothing else."

def evaluate_rag_response(question: str, context: str, answer: str) -> dict:
    """LLM-as-judge: evaluate RAG output on faithfulness and relevance."""
    eval_prompt = f"""Score the following on 1-5.

Question: {question}

Retrieved Context:
{context[:800]}

Generated Answer:
{answer}

Return JSON only:
{{
  "faithfulness": <1-5>,       // Is the answer supported by the context?
  "answer_relevance": <1-5>,   // Does the answer address the question?
  "reasoning": "<1 sentence>"
}}"""
    raw = ask(eval_prompt, system=JUDGE_SYSTEM)
    return json.loads(raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```"))

# Run evaluation on one example
question = "What is RAG and who invented it?"
context_text = format_chunks(retrieve(question))
answer = rag_answer(question)

scores = evaluate_rag_response(question, context_text, answer)
print(f"Question: {question}")
print(f"Answer: {answer[:200]}...")
print(f"\nEvaluation scores:")
print(json.dumps(scores, indent=2))

---
## 7. Common Failure Modes & Fixes

| Problem | Symptom | Fix |
|---------|---------|-----|
| Wrong chunks retrieved | Irrelevant context in answer | Smaller chunks, better embeddings |
| Missing information | "I don't have info about X" (but you do) | Increase `k`, add query expansion |
| Hallucination | Claims facts not in context | Stricter prompt; lower temperature on models that accept it |
| Slow retrieval | High latency | Use HNSW index, cache frequent queries |
| Stale data | Outdated answers | Incremental indexing pipeline |

---

## 🧪 Exercises

1. **Add your own documents** — load a text file and index it. Ask 5 questions about it.
2. **Compare k values** — try `k=1`, `k=3`, `k=10`. How does answer quality change?
3. **Test the failure case** — ask a question that partially overlaps with your data. Does the model hallucinate or admit it doesn't know?
4. **Try HyDE** — generate a hypothetical answer to the question, then use *that* as the search query instead of the original question.

---
**Next:** [Module 03 — Fine-Tuning](../fine-tuning/README.md)

---
## 🧪 Exercises

1. **Chunk Size**: Index the same document at 128, 256, 512 tokens. Which gives best retrieval?
2. **HyDE vs Standard**: Compare standard retrieval vs HyDE on 10 queries.
3. **Re-ranking**: Add cross-encoder re-ranker. Measure precision@3 before/after.